# brier - M5.4 benchmark: raw vs L0 vs L1 vs L2 on banking20

Same protocol, data split and pins as the M3.4 / M4 runs (`docs/results.md`), plus the
**L2 hidden-state head** at label budgets 100 / 200 / 300 (L2 needs at least 5 labels per
class; banking20 has 20 classes). For each budget a class-balanced subsample of the 300-item
calibration split selects layer, solver (ridge / LDA) and alpha by 5-fold out-of-fold NLL and
fits the L2 temperature; the head is scored on the 2,603 test items.

**Runtime:** *Runtime -> Change runtime type -> A100 GPU*. About 15-20 min for Qwen3-1.7B
(L2 adds hidden states for ~5,800 prompts, cached). Section 6 optionally repeats it on Qwen3-4B
(about 35-45 min).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Configuration and install

In [ ]:
BRIER_COMMIT = "8f267218ee92676516f18cf435398abebc1d4d11"  # main after PR #19 (L2 + cache)
MODEL = "Qwen/Qwen3-1.7B"
REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"  # same pinned model as M3.4 / M4
DTYPE = "bfloat16"  # A100/L4 support bf16; on a T4 use "float16"
BATCH = 64
BUDGETS = "100,200,300"
OUT = "/content/results"

In [ ]:
!git clone -q https://github.com/mohdUwaish59/brier.git /content/brier
!git -C /content/brier checkout -q {BRIER_COMMIT}
!pip install -q -e "/content/brier[hf]"
!python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())"

## 3. Smoke run (L2 path only, smallest budget)

L2 cannot run on a small `--limit` (it needs >= 100 calibration labels), so this checks the L2
path end to end with the full calibration split and budget 100 before the long run.

In [ ]:
!python -m brier.bench run --model {MODEL} --revision {REVISION} --dtype {DTYPE} \
    --batch-size {BATCH} --levels L2 --l2-budgets 100 --n-resamples 100 \
    --out /content/results_smoke

In [ ]:
import glob
import json

with open(glob.glob("/content/results_smoke/*.json")[0]) as f:
    smoke = json.load(f)
entry = smoke["l2_curve"]["100"]
print(
    "L2 smoke:", {k: entry[k] for k in ("layer", "solver", "alpha", "temperature", "oof_accuracy")}
)
print(
    "accuracy",
    round(entry["metrics"]["accuracy"][0], 3),
    "| ece",
    round(entry["metrics"]["ece"][0], 3),
)
print("wall seconds:", smoke["timing"]["wall_seconds"])

## 4. Full run (raw, L0, L1, L2)

Keep this tab open. Results are written after each level; the hidden states from the smoke run
are reused from the cache.

In [ ]:
!python -m brier.bench run --model {MODEL} --revision {REVISION} --dtype {DTYPE} \
    --batch-size {BATCH} --levels raw,L0,L1,L2 --l2-budgets {BUDGETS} --out {OUT}

## 5. Results

In [ ]:
import glob
import json

with open(glob.glob(f"{OUT}/*.json")[0]) as f:
    res = json.load(f)


def fmt(v):
    return f"{v[0]:.3f} [{v[1]:.3f}, {v[2]:.3f}]"


rev = res["model"]["revision"] or "unpinned"
print(f"model {res['model']['id']}@{rev[:8]}  brier {res['git_commit']}")
print(
    f"n_test={res['n_test']} n_pool={res['n_pool']} T_L1={res['calibration']['L1']['temperature']:.3f}"
)
print()
print("| metric | raw | L0 | L1 | L2 (300) | L2 - L1 (paired) |")
print("|---|---|---|---|---|---|")
for k in res["levels"]["raw"]:
    cols = [fmt(res["levels"][lv][k]) for lv in ("raw", "L0", "L1", "L2")]
    print(f"| {k} | {' | '.join(cols)} | {fmt(res['comparisons']['L2_minus_L1'][k])} |")
print()
print("| labels | layer | solver | alpha | T | accuracy | ece | nll |")
print("|---|---|---|---|---|---|---|---|")
for budget, e in res["l2_curve"].items():
    m = e["metrics"]
    print(
        f"| {budget} | {e['layer']} | {e['solver']} | {e['alpha']} | {e['temperature']:.3f} | "
        f"{fmt(m['accuracy'])} | {fmt(m['ece'])} | {fmt(m['nll'])} |"
    )
print()
print("wall seconds:", res["timing"]["wall_seconds"])

## 6. Optional: repeat on Qwen3-4B

In [ ]:
MODEL = "Qwen/Qwen3-4B"
REVISION = "1cfa9a7208912126459214e8b04321603b3df60c"  # pinned Qwen3-4B commit
OUT = "/content/results_4b"

In [ ]:
!python -m brier.bench run --model {MODEL} --revision {REVISION} --dtype {DTYPE} \
    --batch-size {BATCH} --levels raw,L0,L1,L2 --l2-budgets {BUDGETS} --out {OUT}

## 7. Download all results (JSON + per-item `.npz`, no state text)

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/brier_m54_results", "zip", "/content", "results")
files.download("/content/brier_m54_results.zip")
try:
    shutil.make_archive("/content/brier_m54_results_4b", "zip", "/content", "results_4b")
    files.download("/content/brier_m54_results_4b.zip")
except FileNotFoundError:
    print("4B run not present (section 6 skipped)")